# Project: Netflix and the Covid crisis 🎬

On 11 March 2020, the World Health Organization declared Covid-19 a pandemic, and lockdowns started all over the world. People stayed home... and watched streaming services. The Netflix stock price jumped.

A time series model learns patterns from the past. What happens when something new and unpredictable occurs? In this project, you will:
* forecast the Netflix stock price with Prophet, using only data from before the crisis;
* measure how wrong this forecast is;
* retrain the model with one, then two months of crisis data, and see how fast it adapts.

In [ ]:
import logging
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from prophet import Prophet
from prophet.plot import plot_components_plotly
from sklearn.metrics import mean_absolute_percentage_error

logging.getLogger("cmdstanpy").disabled = True  # hide the messages of Prophet's optimizer


def plot_forecast(train, test, forecast, title):
    """Plot the training data, the forecast with its uncertainty interval, and the real values."""
    fig = go.Figure([
        go.Scatter(x=forecast["ds"], y=forecast["yhat_upper"], line=dict(width=0), showlegend=False, hoverinfo="skip"),
        go.Scatter(x=forecast["ds"], y=forecast["yhat_lower"], line=dict(width=0), fill="tonexty",
                   fillcolor="rgba(59, 71, 196, 0.2)", name="uncertainty interval"),
        go.Scatter(x=train["ds"], y=train["y"], mode="lines", name="training data", line=dict(color="#999999")),
        go.Scatter(x=forecast["ds"], y=forecast["yhat"], mode="lines", name="forecast", line=dict(color="#3B47C4")),
        go.Scatter(x=test["ds"], y=test["y"], mode="lines", name="real price", line=dict(color="#E4572E")),
    ])
    fig.update_layout(title=title, template="plotly_white", xaxis_range=["2019-06-01", "2020-09-01"])
    return fig

import plotly.io as pio
import plotly.graph_objects as go
# colour palette of the course
pio.templates["course"] = go.layout.Template(
    layout_colorway=["#3B47C4", "#E4572E", "#2E7D4F", "#F2C14E", "#8E7CC3", "#4BA3C3", "#111111", "#999999"]
)
pio.templates.default = "plotly_white+course"

## The data
`data/netflix_stock.csv` contains the daily prices of the Netflix stock (`NFLX`) from January 2017 to August 2020 (source: Yahoo Finance). The prices are adjusted for the stock splits that happened later (Netflix split its stock 10-for-1 in 2025), so they look ten times smaller than the prices quoted in 2020: what matters here is the shape of the curve.

1. Load the data and plot the closing price (`Close`). Add a vertical line on 11 March 2020.

In [ ]:
data = pd.read_csv("data/netflix_stock.csv", index_col="Date", parse_dates=True)
PANDEMIC = "2020-03-11"

fig = px.line(data, y="Close", title="Netflix stock price (closing price)")
fig.add_vline(x=pd.Timestamp(PANDEMIC).timestamp() * 1000, line_dash="dash")
fig.show()

2. Prophet needs a DataFrame with two columns: `ds` (the dates) and `y` (the values to forecast). Create it from the closing price.

In [ ]:
df = pd.DataFrame({"ds": data.index, "y": data["Close"].values})
df.head()

,ds,y
0,2017-01-03,12.749
1,2017-01-04,12.941
2,2017-01-05,13.181
3,2017-01-06,13.107
4,2017-01-09,13.095


## Forecasting the crisis from the past
3. Train a Prophet model on the days **before** 11 March 2020. Keep the next 42 trading days (about two months) as the test set, and forecast them.

💡 The stock market is closed at night: there is one value per day, so use `Prophet(daily_seasonality=False)`.

In [ ]:
train = df[df["ds"] < PANDEMIC]
test = df[df["ds"] >= PANDEMIC].head(42)

model = Prophet(daily_seasonality=False)
model.fit(train)
forecast = model.predict(test[["ds"]])
forecast[["ds", "yhat", "yhat_lower", "yhat_upper"]].head()

,ds,yhat,yhat_lower,yhat_upper
0,2020-03-11,38.450410,36.393433,40.779737
1,2020-03-12,38.527494,36.277017,40.576910
2,2020-03-13,38.500984,36.170310,40.561369
3,2020-03-16,38.521557,36.374231,40.911837
4,2020-03-17,38.681893,36.509350,40.867757


4. Plot the forecast and the real prices of the test period with the `plot_forecast` function (defined in the first cell), and compute the MAPE (mean absolute percentage error) on the test set.

In [ ]:
fig = plot_forecast(train, test, forecast, "Forecast trained before the pandemic")
fig.show()

mape = mean_absolute_percentage_error(test["y"], forecast["yhat"])
print(f"MAPE on the next two months: {mape:.1%}")

MAPE on the next two months: 8.3%


5. Look at the components of the model with `plot_components_plotly`. What did the model learn from the past?

In [ ]:
history_and_test = model.predict(pd.concat([train, test])[["ds"]])
fig = plot_components_plotly(model, history_and_test)
fig.show()

**The model learned a trend made of straight segments, plus small weekly and yearly patterns. Nothing in the past announced a pandemic: the model can only extend the trend it has seen.**

## Learning from the crisis
6. Retrain the model with one month, then two months of crisis data (cut-off dates: 11 April and 11 May 2020). Each time, forecast the next 42 trading days and compute the MAPE.

💡 Write a function, so you don't repeat the same code three times!

In [ ]:
def forecast_after(cutoff, horizon=42):
    """Train on the days before `cutoff`, forecast the next `horizon` trading days."""
    train = df[df["ds"] < cutoff]
    test = df[df["ds"] >= cutoff].head(horizon)
    model = Prophet(daily_seasonality=False).fit(train)
    forecast = model.predict(test[["ds"]])
    return model, forecast, test, mean_absolute_percentage_error(test["y"], forecast["yhat"])


results = {}
for cutoff in ["2020-03-11", "2020-04-11", "2020-05-11"]:
    model, forecast, test, mape = forecast_after(cutoff)
    results[cutoff] = mape
    train = df[df["ds"] < cutoff]
    fig = plot_forecast(train, test, forecast, f"Trained until {cutoff}: MAPE {mape:.1%}")
    fig.show()

pd.Series(results, name="MAPE on the next 42 days").map("{:.1%}".format)

2020-03-11    8.3%
2020-04-11    9.9%
2020-05-11    4.2%
Name: MAPE on the next 42 days, dtype: str

**Conclusion**

* Trained before the crisis, the forecast is off by about 8%: the model extends the past trend and misses the jump.
* One month into the crisis, the forecast is **not better** (about 10%): the model has just seen a sudden drop and a strong rebound, and cannot tell which one will last.
* With two months of crisis data, the new trend is clear and the error falls to about 4%.

What to remember:
* a time series model extends the patterns of the past: it cannot foresee a shock that never happened before;
* after a shock, a model needs time (that is, data) to learn the new situation;
* models used in production must be **monitored** and retrained when the world changes.

💡 Prophet can also use extra variables (`add_regressor`). Be careful: the model needs their values **for the future dates too**. Using, for instance, the real Disney stock price of the test period would be cheating, because nobody knows it at the time of the forecast. Good regressors are known in advance: holidays, planned promotions, the dates of a lockdown once it is announced...